In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 시간-주파수 분석: STFT 와 웨이블릿 변환

## 학습 목표
- 주파수가 시간에 따라 변하는 신호에서 FFT 의 한계와 STFT 의 슬라이딩 윈도우 원리를 살펴본다.
- `scipy.signal.spectrogram` 의 `nperseg`, `noverlap`, `nfft` 파라미터가 결과에 미치는 영향을 비교한다.
- CWT(Morlet)의 다중해상도 특성과 DWT 에 의한 디노이징을 다룬다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from utils import fft

---

## 실습 1. chirp 와 FFT 의 한계

시간에 따라 주파수가 변하는 **chirp 신호**를 만들고 일반 FFT 를 적용해 본다. FFT 는 전 구간의 주파수 성분을 한꺼번에 보여 주므로 **어느 시점에 어떤 주파수가 있었는지**는 알 수 없다. 이것이 시간-주파수 분석이 필요한 이유이다.

In [ ]:
# 선형 chirp 생성: 5 Hz -> 200 Hz, fs=1000 Hz, 2초
fs = 1000
T = 2.0
t = np.arange(0, T, 1 / fs)
x = signal.chirp(t, f0=5, f1=200, t1=T, method='linear')

# FFT (단측 진폭 스펙트럼)
f_axis, A = fft(x, fs)

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(t, x, color='C0', linewidth=0.7)
axes[0].set_title('Chirp Signal x(t): 5 Hz -> 200 Hz (linear sweep)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlim([0, T])

axes[1].plot(f_axis, A, color='C1', linewidth=1.0)
axes[1].set_title('FFT Result: energy spread 5~200 Hz — time info lost')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim([0, 250])
plt.tight_layout()
plt.show()

FFT 결과에서는 에너지가 5 ~ 200 Hz 의 전 구간에 고르게 분포할 뿐이므로, **어느 시점에** 저주파 성분과 고주파 성분이 있었는지 알 수 없다.

### 직접 해보기
1. `f1=500` 으로 끝 주파수를 높이면 FFT 스펙트럼은 어떻게 변하는가?
2. `f0=f1=100` 으로 주파수가 일정한 신호를 만들면 스펙트럼에 하나의 성분이 나타나는가?

---

## 실습 2. 슬라이딩 윈도우 원리

STFT 는 신호를 **짧은 프레임**으로 잘라 각 프레임에 윈도우를 곱한 뒤 FFT 를 수행한다. Hann 윈도우는 프레임의 양 끝을 부드럽게 감쇠시켜 스펙트럼 누설을 줄인다.

In [ ]:
nperseg = 256
hop = 128  # 겹침 50 %
win = signal.windows.hann(nperseg)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x, color='gray', linewidth=0.5, alpha=0.6, label='Original')

frame_colors = ['C0', 'C1', 'C2', 'C3', 'C4']
positions = [i * hop for i in range(5)]
for i, pos in enumerate(positions):
    if pos + nperseg > len(x):
        break
    tw = t[pos:pos + nperseg]
    xw = x[pos:pos + nperseg] * win
    ax.fill_between(tw, -1.3, 1.3, color=frame_colors[i], alpha=0.08)
    ax.plot(tw, xw, color=frame_colors[i], linewidth=1.3,
            label=f'Frame {i+1}')

ax.set_title(f'STFT Sliding Window (nperseg={nperseg}, hop={hop}, Hann)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_xlim([0, 1.5])
ax.set_ylim([-1.4, 1.4])
ax.legend(loc='lower right', ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

각 프레임에는 Hann 윈도우가 적용되어 가운데의 진폭이 크고 양 끝의 진폭이 작다. 이렇게 윈도우를 곱한 각 프레임에서 독립적으로 FFT 를 수행한다.

---

## 실습 3. 첫 스펙트로그램

프레임마다 구한 FFT 결과를 시간 축과 주파수 축의 2차원 평면에 펼친 것이 **스펙트로그램**이다. chirp 신호는 주파수가 시간에 따라 증가하는 성분이므로 밝은 대각선으로 나타난다.

In [ ]:
nperseg = 128
noverlap = nperseg // 2
nfft = nperseg

f, t_spec, Sxx = signal.spectrogram(
    x, fs=fs, nperseg=nperseg, noverlap=noverlap, nfft=nfft, window='hann', detrend=False)

fig, ax = plt.subplots(figsize=(10, 5))
pcm = ax.pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12),
                    shading='gouraud', cmap='viridis')
ax.set_title(f'Chirp Spectrogram (nperseg={nperseg}, noverlap={noverlap})')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_ylim([0, 250])
fig.colorbar(pcm, ax=ax, label='Power (dB)')
plt.tight_layout()
plt.show()

밝은 대각선을 따라 주파수가 5 Hz 에서 200 Hz 까지 증가하는 것이 보인다. 이제 **"언제" 어떤 주파수가 있었는지**를 직접 볼 수 있다.

### 직접 해보기
1. `nperseg=64` 로 줄이면 시간 축은 촘촘해지지만 주파수 축은 거칠어지는가?
2. `window='hamming'` 으로 바꾸면 결과에 눈에 띄는 차이가 있는가?

---

## 실습 4. `nperseg` 변화: 시간-주파수 분해능의 트레이드오프

윈도우 길이 `nperseg` 가 길어질수록 **주파수 분해능**(Δf = fs/nperseg)은 좋아지지만 시간 분해능은 나빠진다. 세 값을 나란히 비교한다.

In [ ]:
nperseg_list = [32, 128, 256]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, N in zip(axes, nperseg_list):
    f, t_spec, Sxx = signal.spectrogram(
        x, fs=fs, nperseg=N, noverlap=N // 2, nfft=N, window='hann', detrend=False)
    pcm = ax.pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12),
                        shading='gouraud', cmap='viridis')
    ax.set_title(f'nperseg={N}  (Δf={fs/N:.2f} Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_ylim([0, 250])

axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(pcm, ax=axes.tolist(), label='Power (dB)', pad=0.02)
plt.show()

- `nperseg=32`: 주파수 분해능이 부족하여 chirp 의 대각선이 주파수 방향으로 넓게 퍼진다.
- `nperseg=256`: 윈도우 길이 0.256 s 동안 chirp 의 주파수가 약 25 Hz 변하므로, 긴 윈도우의 주파수 분해능(3.9 Hz)은 이 주파수 변화에 가려진다. 대각선의 두께는 128일 때보다 조금 얇아질 뿐이며, 주파수가 변하지 않는 톤이라면 절반이 된다. 또 시간 분해능이 부족하여 프레임 간격(128 ms)마다 대각선의 밝기가 끊어져 점선과 같은 모양이 된다.
- `nperseg=128`: 시간 분해능과 주파수 분해능이 균형을 이룬다.

### 직접 해보기
`nperseg_list` 에 `512` 를 추가해 보라. 주파수 분해능은 더 좋아지지만 chirp 의 시작과 끝이 어떻게 번지는지 확인한다.

---

## 실습 5. 겹침 `noverlap` 변화

`noverlap` 을 늘리면 프레임 사이의 겹침이 커져 시간 축이 매끄러워진다. 일반적으로 **50 ~ 75 %를** 권장한다.

In [ ]:
nperseg = 256
overlap_ratios = [0.25, 0.50, 0.95]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, r in zip(axes, overlap_ratios):
    noverlap = int(nperseg * r)
    f, t_spec, Sxx = signal.spectrogram(
        x, fs=fs, nperseg=nperseg, noverlap=noverlap,
        nfft=nperseg, window='hann', detrend=False)
    pcm = ax.pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12),
                        shading='gouraud', cmap='viridis')
    ax.set_title(f'overlap={int(r*100)}%  (hop={nperseg - noverlap})')
    ax.set_xlabel('Time (s)')
    ax.set_ylim([0, 250])

axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(pcm, ax=axes.tolist(), label='Power (dB)', pad=0.02)
plt.show()

겹침이 25 %에서 95 %로 증가할수록 시간 축 방향의 계단 모양이 사라지고 대각선이 매끄러워진다. 일반적으로 **50 ~ 75 %가** 적절한 선택이다.

---

## 실습 6. `nfft` 변화 (제로패딩 효과)

`nfft > nperseg` 로 두면 FFT 길이가 제로패딩으로 늘어난다. 그림은 부드러워지지만 **실제 주파수 분해능은 nperseg 가 결정**한다.

In [ ]:
nperseg = 32
noverlap = 16
nfft_list = [32, 64, 320]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, nfft in zip(axes, nfft_list):
    f, t_spec, Sxx = signal.spectrogram(
        x, fs=fs, nperseg=nperseg, noverlap=noverlap,
        nfft=nfft, window='hann', detrend=False)
    pcm = ax.pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12),
                        shading='gouraud', cmap='viridis')
    zp = 'none' if nfft == nperseg else f'{nfft - nperseg} pts'
    ax.set_title(f'nfft={nfft}  (zero-padding: {zp})')
    ax.set_xlabel('Time (s)')
    ax.set_ylim([0, 250])

axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(pcm, ax=axes.tolist(), label='Power (dB)', pad=0.02)
plt.show()

`nfft` 를 키우면 그림은 부드러워 보이지만, `nperseg=32` 가 정하는 31.25 Hz 의 **실제** 분해능 한계는 변하지 않는다. 세 패널에서 대각선의 두께가 같다.

### 직접 해보기
`nperseg=128`, `noverlap=64` 로 바꾸고 `nfft_list = [128, 256, 1280]` 으로 비교하여, 대각선이 얇아지는 것이 `nfft` 가 아니라 `nperseg` 때문이라는 것을 확인한다.

---

## 실습 7. 실제 오디오: knocking.mp3

실제 음원에서도 STFT 파라미터가 결과의 품질을 좌우한다. 엔진 노킹 음원 `knocking.mp3` 에서는 큰 충격이 약 77 ms 주기로 반복되고, 그 사이에 작은 충격이 하나씩 있다. 이 음원을 불러와 파형을 보고, 주파수 분해능이 부족한 **Poor 파라미터**(64 pt, 겹침 10 %)와 **Good 파라미터**(1024 pt, 겹침 75 %)를 나란히 비교한다. Good 파라미터에서는 각 충격의 공진 대역이 뚜렷하게 나타난다.

In [ ]:
try:
    import librosa
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

# 오디오 로드 (sr=None 이면 파일 원본 샘플링 레이트 유지)
audio, fs_audio = librosa.load('./data/knocking.mp3', sr=None, mono=True)
print(f'Sampling Frequency: {fs_audio} Hz, Duration: {len(audio)/fs_audio:.2f} s')

t_audio = np.arange(len(audio)) / fs_audio
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_audio, audio, color='C0', linewidth=0.4)
ax.set_title('knocking.mp3 — Time Waveform')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_xlim([0, t_audio[-1]])
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import spectrogram as _spec

# 두 파라미터 세트
configs = [
    ('Poor  (nperseg=64,   overlap=10%, nfft=64)',   64,   int(64*0.10),   64),
    ('Good  (nperseg=1024, overlap=75%, nfft=1024)', 1024, int(1024*0.75), 1024),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
vmax_lin = None
spec_results = []
for nps, nov, nf in [(c[1], c[2], c[3]) for c in configs]:
    f_k, t_k, S_k = _spec(audio, fs=fs_audio, nperseg=nps, noverlap=nov, nfft=nf, window='hann')
    S_mag = np.sqrt(S_k)   # amplitude (linear)
    spec_results.append((f_k, t_k, S_mag))
    vmax_lin = max(vmax_lin, S_mag.max()) if vmax_lin else S_mag.max()

for ax, (label, _, _, _), (f_k, t_k, S_mag) in zip(axes, configs, spec_results):
    pcm = ax.pcolormesh(t_k, f_k, S_mag, shading='gouraud', cmap='viridis',
                        vmin=0, vmax=vmax_lin * 0.5)
    ax.set_ylim(0, fs_audio/2)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
    ax.set_title(label)
    fig.colorbar(pcm, ax=ax, label='|X|')

fig.suptitle('knocking.mp3 — STFT parameter quality (linear scale)')
plt.tight_layout(); plt.show()

**관찰**:
- **Poor (nperseg=64, overlap 10%)**: 윈도우가 약 1.5 ms 로 짧아 충격이 발생한 시각은 가는 세로 줄무늬로 뚜렷하게 표시된다. 그러나 $\Delta f = f_s/64 \approx 690$ Hz 이므로 각 충격의 에너지가 주파수 방향으로 약 3 ~ 15 kHz 에 걸쳐 넓게 퍼진다. 세로 줄무늬 안에서 4 ~ 6 kHz 와 7.5 ~ 8.5 kHz 가 밝은 것까지는 보이지만, Good 에서 분리되는 7.2 kHz, 8 kHz 같은 개별 공진은 한 빈 안에 합쳐진다.
- **Good (nperseg=1024, overlap 75%)**: $\Delta f \approx 43$ Hz 로 주파수 분해능이 높다. 그래서 각 충격의 공진 성분이 4 ~ 6 kHz 와 7 ~ 8.5 kHz 의 밝은 점으로 분리되어 충격마다 같은 주파수에 나타나며, 가장 밝은 점은 약 7.2 kHz 와 8 kHz 에 있다. 1 ~ 2 kHz 의 밝은 점은 충격 사이에도 비슷한 세기로 있으므로 충격에 의한 공진이 아니다. 대신 윈도우가 약 23 ms 이므로 충격 하나가 시간 축 방향으로 그만큼 퍼진다. 큰 충격이 약 77 ms 마다 반복되는 **주기적인 충격**의 구조는 두 설정 모두에서 뚜렷하다.
- 실제 분석에서는 신호의 특성, 곧 **충격성 신호인지 정상 신호인지**에 맞추어 nperseg 를 신호의 주요 시간 척도와 비슷하게 두고, 겹침은 50 ~ 75 %로 정한다.

---

## 실습 8. CWT: 연속 웨이블릿 변환

CWT 는 분석 주파수에 대응하는 스케일에 따라 윈도우 길이를 **자동으로 조정**한다. 저주파에서는 주파수 분해능이, 고주파에서는 시간 분해능이 상대적으로 더 좋으며, 이것이 다중해상도 분석(multi-resolution analysis)의 특성이다.

In [ ]:
try:
    import pywt
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'PyWavelets'])
    import pywt

# 실습 1 의 chirp 신호를 그대로 사용 (fs=1000 Hz, 5->200 Hz, 2초)
wavelet = 'cmor2.0-0.7958'   # complex Morlet (omega_c = 5)
fc = pywt.central_frequency(wavelet)
dt = 1.0 / fs

# 분석할 주파수 범위: 1~250 Hz → 로그 간격 스케일
f_min, f_max = 1.0, 250.0
scale_min = fc / (f_max * dt)
scale_max = fc / (f_min * dt)
scales = np.logspace(np.log10(scale_min), np.log10(scale_max), 150)

coefs, freqs_cwt = pywt.cwt(x, scales, wavelet, sampling_period=dt)
power = np.abs(coefs) ** 2
print(f'CWT coefficients shape: {coefs.shape}  (scales × samples)')

In [ ]:
# CWT vs STFT(nperseg=128) 나란히 비교: 두 패널 모두 dB, 같은 동적 범위
nperseg = 128
dyn_range = 30   # dB
f_stft, t_stft, Sxx = signal.spectrogram(
    x, fs=fs, nperseg=nperseg, noverlap=nperseg * 3 // 4,
    nfft=nperseg, window='hann', detrend=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pcm0 = axes[0].pcolormesh(t_stft, f_stft, 10 * np.log10(Sxx + 1e-12),
                          shading='gouraud', cmap='viridis',
                          vmin=10 * np.log10(Sxx.max()) - dyn_range)
axes[0].set_title(f'STFT (nperseg={nperseg}, Hann)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_ylim([0, 250])
fig.colorbar(pcm0, ax=axes[0], label='Power (dB)')

# 선형 색이면 고주파 쪽 CWT 능선이 최댓값의 13 ~ 17 %여서 거의 보이지 않으므로 dB 로 그린다
pcm1 = axes[1].pcolormesh(t, freqs_cwt, 10 * np.log10(power + 1e-12),
                          shading='gouraud', cmap='viridis',
                          vmin=10 * np.log10(power.max()) - dyn_range)
axes[1].set_title('CWT (complex Morlet), log-spaced scales')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_ylim([0, 250])
fig.colorbar(pcm1, ax=axes[1], label='Power (dB)')

plt.tight_layout()
plt.show()

CWT 패널에서 chirp 성분의 능선은 저주파에서 좁고 고주파로 갈수록 넓어진다. 능선의 3 dB 폭은 25 Hz 부근에서 약 10 Hz, 150 Hz 부근에서 약 50 Hz 이다. 즉 주파수 분해능이 분석 주파수에 비례하여 나빠진다. 그 대신 고주파에서는 웨이블릿이 짧아 시간 분해능이 좋아지지만, chirp 처럼 주파수가 계속 변하는 신호에서는 이 효과가 능선의 모양으로 보이지 않는다. 이처럼 주파수에 따라 분해능의 배분이 바뀌는 것을 **다중해상도 분석**(multi-resolution analysis)이라고 한다. CWT 패널의 아래 끝 1 ~ 5 Hz 에 보이는 밝은 가로 방향의 무늬와 그 위 약 50 Hz 까지 흩어진 흐린 무늬는 chirp 성분(5 Hz 에서 시작)이 아니며, `pywt.cwt` 가 큰 스케일(낮은 주파수)에서 계산하는 과정에서 생긴 무늬이다.

### 직접 해보기
`scales` 를 `np.logspace(np.log10(scale_min), np.log10(scale_max), 50)` 으로 줄이면 세로축인 주파수 축의 샘플링이 어떻게 거칠어지는지 확인한다.

---

## 실습 9. DWT 기반 디노이징

DWT 로 신호를 여러 스케일로 분해하면 저주파 신호는 **근사 계수**(cA)에 모이고, 광대역 잡음은 모든 **세부 계수**(cD)에 분포한다. 여기에 **표준 VisuShrink** 를 적용한다. 가장 미세한 세부 계수에서 중앙값 절대편차(MAD)로 잡음 표준편차를 추정하고, **범용 임계값(universal threshold)** $\lambda = \hat\sigma\sqrt{2\ln N}$ 으로 **모든 세부 계수에 일괄하여** 연성(soft) 임계처리를 적용한다. 작은 잡음 계수는 0이 되고 큰 신호 계수는 남으므로, 어느 대역이 잡음인지 미리 알지 않아도 잡음이 제거된다.

신호는 5, 15, 40 Hz 의 세 정현파에 $t = 0.5$ s 의 짧은 150 Hz 버스트를 더하고, 표준편차 $\sigma = 0.4$ 의 가우시안 잡음을 섞은 것이다($f_s = 1000$ Hz, $N = 1000$). 버스트의 진폭은 2.0이고 가우시안 포락선의 표준편차는 20 ms(반값 폭 약 47 ms)이다. 버스트는 세부 계수 cD2(125 ~ 250 Hz)에 **소수의 큰 계수**로 나타나므로, 임계값을 넘어 남는 계수가 무엇인지 볼 수 있다.

In [ ]:
# 합성 신호: 5 + 15 + 40 Hz 정현파 + 짧은 150 Hz 버스트 + 가우시안 잡음
fs_dn = 1000
T_dn = 1.0
t_dn = np.arange(0, T_dn, 1 / fs_dn)
N = len(t_dn)

burst = 2.0 * np.sin(2 * np.pi * 150 * t_dn) * np.exp(-0.5 * ((t_dn - 0.5) / 0.02) ** 2)
x_clean = (np.sin(2 * np.pi * 5 * t_dn)
           + 0.5 * np.sin(2 * np.pi * 15 * t_dn)
           + 0.8 * np.sin(2 * np.pi * 40 * t_dn)
           + burst)
sigma = 0.4
rng = np.random.default_rng(42)
x_noisy = x_clean + sigma * rng.standard_normal(N)

# DWT 3레벨 분해 (db8)
#   fs=1000, level=3 이론적 대역:
#     cA3 (0 ~ 62.5 Hz)   : 5, 15, 40 Hz 정현파가 모두 이 근사 대역에 있음
#     cD3 (62.5 ~ 125 Hz) : 잡음
#     cD2 (125 ~ 250 Hz)  : 잡음 + 150 Hz 버스트 (짧은 구간의 큰 계수)
#     cD1 (250 ~ 500 Hz)  : 잡음
wavelet = 'db8'
level = 3
coeffs = pywt.wavedec(x_noisy, wavelet, level=level)
cA3, cD3, cD2, cD1 = coeffs

# 각 레벨의 계수 통계: cA3(신호)의 std 가 가장 크고, cD2 는 버스트 때문에 max 가 크다
for name, coef in [('cA3', cA3), ('cD3', cD3), ('cD2', cD2), ('cD1', cD1)]:
    print(f'  {name}: len={len(coef):4d}, std={np.std(coef):.3f}, max|.|={np.max(np.abs(coef)):.3f}')

# 잡음 크기(표준편차) 추정: 잡음이 지배적인 가장 미세한 세부 계수(cD1)에서 추정한다.
#   median(|cD1|)/0.6745 : 가우시안에서 중앙값 절댓값과 sigma 의 관계로 sigma 를 환산.
sigma_hat = np.median(np.abs(cD1)) / 0.6745
# 임계값: 잡음 계수가 거의 넘지 못하는 높이 = sigma * sqrt(2 ln N)
lam = sigma_hat * np.sqrt(2 * np.log(N))
print(f'Estimated noise sigma = {sigma_hat:.4f}  (true sigma = {sigma})')
print(f'Universal threshold lambda = {lam:.4f}')
for name, coef in [('cD3', cD3), ('cD2', cD2), ('cD1', cD1)]:
    print(f'  {name}: coefficients above lambda = {np.sum(np.abs(coef) > lam)}')

# 근사 계수(cA3, 신호)는 그대로 두고, 세부 계수(잡음)에만 연성(soft) 임계처리 적용.
# 임계값보다 작은 계수는 0, 큰 계수는 임계값만큼 줄여서 복원한다.
coeffs_thr = [cA3] + [pywt.threshold(c, lam, mode='soft') for c in (cD3, cD2, cD1)]
x_denoised = pywt.waverec(coeffs_thr, wavelet)[:N]

# SNR 비교
def snr_db(clean, meas):
    noise = meas - clean
    return 10 * np.log10(np.mean(clean ** 2) / (np.mean(noise ** 2) + 1e-12))

snr_before = snr_db(x_clean, x_noisy)
snr_after  = snr_db(x_clean, x_denoised)
print(f'SNR: {snr_before:.2f} dB  ->  {snr_after:.2f} dB  (improvement +{snr_after - snr_before:.2f} dB)')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t_dn, x_clean, color='C0', linewidth=1.0)
axes[0].set_title('(a) Original Signal x_clean')
axes[0].set_ylabel('Amplitude')

axes[1].plot(t_dn, x_noisy, color='C1', linewidth=0.6)
axes[1].set_title(f'(b) Noisy Signal x_noisy  (SNR={snr_before:.1f} dB)')
axes[1].set_ylabel('Amplitude')

axes[2].plot(t_dn, x_denoised, color='C2', linewidth=1.0, label='Denoised')
axes[2].plot(t_dn, x_clean, color='C0', linewidth=0.8, alpha=0.5, label='Original')
axes[2].set_title(f'(c) Denoised  (SNR={snr_after:.1f} dB)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Amplitude')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# 디노이징 효과를 스펙트럼에서 확인
f_c, A_c = fft(x_clean,    fs_dn)
f_n, A_n = fft(x_noisy,    fs_dn)
f_d, A_d = fft(x_denoised, fs_dn)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(f_n, A_n, color='C1', alpha=0.4, lw=1.0, label='Noisy')
ax.plot(f_c, A_c, color='C0',             lw=1.5, label='Clean')
ax.plot(f_d, A_d, color='C2',             lw=1.2, ls='--', label='Denoised')
ax.set_xlim(0, 300); ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Amplitude')
ax.axvline(62.5, color='gray', ls=':', lw=1.0, label='cA3 band edge (62.5 Hz)')
ax.set_title('Spectra before and after wavelet denoising')
ax.legend(); fig.tight_layout(); plt.show()

**관찰**:
- 5, 15, 40 Hz 정현파는 모두 근사 계수 cA3(0 ~ 62.5 Hz)에 모이고, 잡음은 세부 계수 전체에 분포한다. cD1 의 MAD 로 추정한 잡음 표준편차는 $\hat\sigma \approx 0.36$ 으로 실제 값 $\sigma = 0.4$ 에 가깝고, 임계값은 $\lambda \approx 1.34$ 이다.
- 임계값을 넘어 남는 세부 계수는 cD3 에 2개, cD2 에 10개, cD1 에 0개이다. cD2 의 10개는 $t = 0.5$ s 의 150 Hz 버스트에 해당하며, 큰 계수는 신호이고 작은 계수는 잡음이라는 VisuShrink 의 논리를 그대로 확인할 수 있다.
- 세 정현파 성분은 보존되고 62.5 Hz 보다 높은 주파수의 잡음 바닥은 낮아지며, SNR 이 약 8.1 dB 에서 12.9 dB 로 개선된다. 0 ~ 62.5 Hz 의 잡음은 cA3 와 함께 남는다. 150 Hz 버스트는 남지만, 연성 임계처리는 임계값을 넘은 계수도 $\lambda$ 만큼 줄이므로 잡음이 없는 신호보다 작게 복원된다. 버스트 스펙트럼의 최댓값은 약 0.10에서 0.04로 낮아진다.
- 85 Hz 에 잡음이 없는 신호에는 없던 작은 성분(약 0.08)이 새로 나타난다. 이 성분은 40 Hz 정현파가 cA3(샘플링 주파수 125 Hz)에 남긴 영상 $125 - 40 = 85$ Hz 이다. 분해한 계수를 그대로 합성하면 cD3 가 이 영상을 상쇄하지만, 임계처리로 cD3 를 거의 0으로 만들었으므로 상쇄가 이루어지지 않았다.

---

## 정리

이 노트북에서는 시간에 따라 변하는 신호를 분석하는 **STFT, CWT, DWT** 의 세 가지 도구를 익혔다.

- **STFT**: 고정된 윈도우를 사용하므로 시간 분해능과 주파수 분해능 사이에 트레이드오프가 있으며, `nperseg`, `noverlap`, `nfft` 로 결과를 조절한다.
- **CWT**: 스케일에 따라 윈도우 길이를 자동으로 조정하는 다중해상도 분석이다.
- **DWT**: 다단계 이진 분해와 연성 임계처리로 디노이징을 수행한다.